In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WORLD_SIZE"] = "1"

In [7]:
from datasets import load_from_disk
import torch
import json
import matplotlib.pyplot as plt
import seaborn as sns
from bert_score import score, BERTScorer
from tqdm.notebook import tqdm

In [5]:
full_dataset = load_from_disk("/gpfs/radev/home/tj372/project/paper_polish/merged_paper_reviews_2025")
full_dataset = full_dataset["test"]
full_dataset

Dataset({
    features: ['model_input', 'result', 'id', 'title', 'track', 'status', 'keywords', 'primary_area', 'author', 'authorids', 'aff', 'aff_domain', 'position', 'rating', 'confidence', 'soundness', 'contribution', 'presentation', 'rating_avg', 'confidence_avg', 'soundness_avg', 'contribution_avg', 'presentation_avg', 'replies_avg', 'corr_rating_confidence', 'project', 'github', 'site', 'or', 'strengths', 'weaknesses', 'summary', 'questions'],
    num_rows: 1162
})

In [6]:
strengths_per_paper = {}
weaknesses_per_paper = {}
for sample in full_dataset:
    paper_id = sample["id"]
    strengths_per_paper[paper_id] = sample["strengths"]
    weaknesses_per_paper[paper_id] = sample["weaknesses"]

print(len(strengths_per_paper))
print(len(weaknesses_per_paper))

1162
1162


In [26]:
model_paths = {     
    "base": "/gpfs/radev/home/ap2853/paper_polish/results/base_gemma3-4b.json",
    "rank32_ckpt1000": "/gpfs/radev/home/ap2853/paper_polish/results/rank32_ckpt1000.json",
    "rank128_ckpt400": "/gpfs/radev/home/ap2853/paper_polish/results/rank128_ckpt400.json",
    "rank128_ckpt1000": "/gpfs/radev/home/ap2853/paper_polish/results/rank128_ckpt1000.json"
}

results = {}

scorer = BERTScorer(device="cuda", lang="en")

for model_name, path in model_paths.items():
    print(f"Processing {model_name}...")
    with open(path, "r") as f:
        data = json.load(f)

    results[model_name] = {"strength": [], "weakness": []}
    for sample in tqdm(data):
        paper_id = sample["paper_id"]
        review_type = sample["review_type"]
        if review_type == "strength":
            gt_reviews = strengths_per_paper[paper_id]
        else:
            gt_reviews = weaknesses_per_paper[paper_id]

        # Compute bert score against each of the gt reviews
        generated_review = sample["generated_review"]
        if not generated_review.strip():
            print("Empty generated review for paper id: ", paper_id)
            continue
        cands = [generated_review] * len(gt_reviews)
        refs = gt_reviews
        _, _, scores = scorer.score(cands, refs)
        results[model_name][review_type].append(max(scores).item())

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Processing base...


  0%|          | 0/100 [00:00<?, ?it/s]

Processing rank32_ckpt1000...


  0%|          | 0/100 [00:00<?, ?it/s]

Empty generated review for paper id:  rIJbFQ1zII
Empty generated review for paper id:  of25Zg4AdM
Empty generated review for paper id:  XMOaOigOQo
Empty generated review for paper id:  XMOaOigOQo
Empty generated review for paper id:  A9y3LFX4ds
Empty generated review for paper id:  50RNY6uM2Q
Empty generated review for paper id:  50RNY6uM2Q
Empty generated review for paper id:  ZS7UEI3vG5
Processing rank128_ckpt400...


  0%|          | 0/100 [00:00<?, ?it/s]

Processing rank128_ckpt1000...


  0%|          | 0/100 [00:00<?, ?it/s]

Empty generated review for paper id:  A9y3LFX4ds
Empty generated review for paper id:  A9y3LFX4ds
Empty generated review for paper id:  A9y3LFX4ds
Empty generated review for paper id:  9qpdDiDQ2H


In [27]:
averages = {}
for model_name in results:
    strengths = results[model_name]["strength"]
    weaknesses = results[model_name]["weakness"]
    all = strengths + weaknesses
    strengths_avg = sum(strengths) / len(strengths)
    weaknesses_avg = sum(weaknesses) / len(weaknesses)
    all_avg = sum(all) / len(all)
    averages[model_name] = {"strength": strengths_avg, "weakness": weaknesses_avg, "all": all_avg}
averages


{'base': {'strength': 0.8311597692966461,
  'weakness': 0.8183366787433625,
  'all': 0.8247482240200043},
 'rank32_ckpt1000': {'strength': 0.8401666581630707,
  'weakness': 0.8277624586354131,
  'all': 0.8339645583992419},
 'rank128_ckpt400': {'strength': 0.8370652091503143,
  'weakness': 0.8279449605941772,
  'all': 0.8325050848722458},
 'rank128_ckpt1000': {'strength': 0.8374648855087605,
  'weakness': 0.8278179947210818,
  'all': 0.8325409516692162}}

In [24]:
sample = data[0]
generated_review = sample["generated_review"]
gt_review = sample["ground_truth"]
score([generated_review], [gt_review], lang="en", verbose=True)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.11 seconds, 9.05 sentences/sec


(tensor([0.8493]), tensor([0.8694]), tensor([0.8592]))

In [20]:
generated_review

'### REVIEW:\n**STRENGTH**\n\nThe authors propose a novel approach to infuse human brain signals into the visual navigation pipeline, offering a fresh perspective for improving the robustness of visual navigation agents.\nThe proposed approach is systematic, incorporating different stages of neural representations from human brains at different levels of the visual processing hierarchy.\nThe authors conduct extensive evaluations, demonstrating significant improvements on the robustness of agent.\nThe approach is versatile and generalizable to broader areas of embodied AI, potentially extending beyond visual navigation to other decision-making and planning problems.\nThe authors provide a detailed breakdown of the different parts of the approach, and each part is evaluated separately, making the study more interpretable and easier to follow.\nThe findings are significant and compelling, as they propose a framework for enhancing AI agents with neural representations from human brains.\n'

In [18]:
sample

{'paper_id': 'HVY6qL2J9L',
 'ground_truth': 'The major strength of the study is an innovative use of integrating brain data into a downstream task. I haven’t seen it does this way for visual navigation tasks.\nThe paper in general is well motivated.\nRobustness is an important problem to solve and using brain data seems like a promising avenue for research.\nThere is also the potential for broader applications of this strategy even beyond navigation.',
 'generated_review': '### REVIEW:\n**STRENGTH**\n\nThe authors propose a novel approach to infuse human brain signals into the visual navigation pipeline, offering a fresh perspective for improving the robustness of visual navigation agents.\nThe proposed approach is systematic, incorporating different stages of neural representations from human brains at different levels of the visual processing hierarchy.\nThe authors conduct extensive evaluations, demonstrating significant improvements on the robustness of agent.\nThe approach is vers